# Urban Mobility Intelligence

## 03 — Feature Engineering

### Objective

Develop numerical and contextual features for identifying
unusual trip and transaction behaviour.

The feature engineering process focuses on:

1. Temporal context
2. Pricing behaviour
3. Journey efficiency
4. Relative deviations from expected behaviour

Synthetic scenario labels are excluded from feature
construction and model inputs to prevent target leakage.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

trips_df = spark.table(
    "workspace.default.urban_mobility_trips_validated"
)

zones_df = spark.table(
    "workspace.default.urban_mobility_zones"
)

print("Total trips:", trips_df.count())

Total trips: 120000


In [0]:
features_df = (
    trips_df
    .withColumn(
        "time_period",
        F.when(
            F.col("pickup_hour").between(7, 9),
            "Morning Peak"
        )
        .when(
            F.col("pickup_hour").between(16, 19),
            "Evening Peak"
        )
        .otherwise("Off-Peak")
    )
    .withColumn(
        "is_peak_period",
        F.when(
            F.col("time_period") != "Off-Peak", 1
        ).otherwise(0)
    )
)

features_df.groupBy(
    "time_period", "is_peak_period"
).count().show()

+------------+--------------+-----+
| time_period|is_peak_period|count|
+------------+--------------+-----+
|    Off-Peak|             0|85136|
|Morning Peak|             1|14950|
|Evening Peak|             1|19914|
+------------+--------------+-----+



In [0]:
# Calculate typical journey characteristics by time period

period_baselines_df = (
    features_df
    .groupBy("time_period")
    .agg(
        F.expr(
            "percentile_approx(average_speed_kmh, 0.5)"
        ).alias("period_median_speed"),

        F.expr(
            "percentile_approx(trip_duration_min, 0.5)"
        ).alias("period_median_duration"),

        F.expr(
            "percentile_approx(fare_per_km_eur, 0.5)"
        ).alias("period_median_fare_per_km")
    )
)

display(period_baselines_df)

time_period,period_median_speed,period_median_duration,period_median_fare_per_km
Off-Peak,20.99,13.4,3.31
Morning Peak,16.11,17.2,3.6
Evening Peak,16.05,17.0,3.62


In [0]:
cutoff = features_df.select(
    F.unix_timestamp("pickup_datetime").alias("pickup_ts")
).approxQuantile("pickup_ts", [0.8], 0.001)[0]

train_df = features_df.filter(
    F.unix_timestamp("pickup_datetime") < cutoff
)

test_df = features_df.filter(
    F.unix_timestamp("pickup_datetime") >= cutoff
)

print("Training trips:", train_df.count())
print("Evaluation trips:", test_df.count())

Training trips: 95912
Evaluation trips: 24088


In [0]:
train_baselines_df = (
    train_df
    .groupBy("time_period")
    .agg(
        F.expr(
            "percentile_approx(average_speed_kmh, 0.5)"
        ).alias("median_speed"),

        F.expr(
            "percentile_approx(trip_duration_min, 0.5)"
        ).alias("median_duration"),

        F.expr(
            "percentile_approx(fare_per_km_eur, 0.5)"
        ).alias("median_fare_per_km")
    )
)

display(train_baselines_df)

time_period,median_speed,median_duration,median_fare_per_km
Off-Peak,21.0,13.4,3.31
Morning Peak,16.07,17.2,3.6
Evening Peak,16.07,16.9,3.62


In [0]:
def add_contextual_features(df, baselines):
    return (
        df.join(baselines, on="time_period", how="left")
        .withColumn(
            "speed_deviation",
            F.col("average_speed_kmh") / F.col("median_speed") - 1
        )
        .withColumn(
            "duration_deviation",
            F.col("trip_duration_min") / F.col("median_duration") - 1
        )
        .withColumn(
            "fare_per_km_deviation",
            F.col("fare_per_km_eur") / F.col("median_fare_per_km") - 1
        )
    )

train_features_df = add_contextual_features(
    train_df, train_baselines_df
)

test_features_df = add_contextual_features(
    test_df, train_baselines_df
)

In [0]:
feature_columns = [
    "trip_distance_km",
    "trip_duration_min",
    "average_speed_kmh",
    "total_amount_eur",
    "fare_per_km_eur",
    "tip_to_fare_ratio",
    "is_peak_period",
    "speed_deviation",
    "duration_deviation",
    "fare_per_km_deviation"
]

train_features_df.select(feature_columns).summary().show()

print("Training rows:", train_features_df.count())
print("Evaluation rows:", test_features_df.count())

+-------+-----------------+------------------+------------------+------------------+------------------+------------------+------------------+--------------------+--------------------+---------------------+
|summary| trip_distance_km| trip_duration_min| average_speed_kmh|  total_amount_eur|   fare_per_km_eur| tip_to_fare_ratio|    is_peak_period|     speed_deviation|  duration_deviation|fare_per_km_deviation|
+-------+-----------------+------------------+------------------+------------------+------------------+------------------+------------------+--------------------+--------------------+---------------------+
|  count|            95912|             95912|             95912|             95912|             95912|             95912|             95912|               95912|               95912|                95912|
|   mean|5.944991450496286| 18.36025314871977|21.932710192676772|20.985269726415897|3.7339142130286125|0.0909430936691975|0.2907039786470932|  0.1206813641433641|  0.2684883442

In [0]:
train_features_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "workspace.default.urban_mobility_train_features"
    )

test_features_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "workspace.default.urban_mobility_test_features"
    )

print("Feature tables saved.")

Feature tables saved.


In [0]:
for table in [
    "urban_mobility_train_features",
    "urban_mobility_test_features"
]:
    df = spark.table(f"workspace.default.{table}")
    print(table, df.count())

urban_mobility_train_features 95912
urban_mobility_test_features 24088


## Feature Engineering Summary

Created 10 numerical modelling features covering journey
characteristics, transaction behaviour and temporal context.

A chronological split produced:
- Training: 95,912 trips (79.93%)
- Evaluation: 24,088 trips (20.07%)

Contextual speed, duration and fare-per-kilometre deviations
were calculated relative to time-period medians estimated
exclusively from training data.

Synthetic scenario labels were excluded from feature
engineering and will be used only for retrospective evaluation.

The resulting datasets were stored as managed Delta tables.